This document covers all classification models available in `neer_match_utilities` for entity matching. The examples assume you have already prepared your data as described in [basic training pipeline](basic_training_pipeline.md).

## Baseline Models

Baseline models (Logit, Probit, GradientBoost) use the `BaselineTrainingPipe` class. They are faster to train and serve as good benchmarks.

### Logit Model

Logistic regression using statsmodels. A simple, interpretable baseline.


In [ ]:
from neer_match_utilities.baseline_training import BaselineTrainingPipe

training_pipeline = BaselineTrainingPipe(
    # Required
    model_name='my_logit_model',
    similarity_map=smap,
    training_data=(left_train, right_train, matches_train),
    testing_data=(left_test, right_test, matches_test),

    # Model type
    model_kind="logit",  # "logit" | "probit" | "gb"

    # ID columns (must match your data)
    id_left_col="company_id",
    id_right_col="company_id",

    # How matches dataframe is structured
    matches_id_left="left",       # column name for left IDs in matches
    matches_id_right="right",     # column name for right IDs in matches
    matches_are_indices=True,     # True if matches contain row indices, False if IDs

    # Sampling: fraction of non-matches to use during fitting
    mismatch_share_fit=1.0,       # 1.0 = use all, 0.1 = use 10%
    random_state=42,
    shuffle_fit=True,
    pairs_per_chunk=250_000,      # pairs whose features are in memory at once

    # Prediction threshold
    threshold=0.5,
    tune_threshold=False,         # Logit/Probit: typically use 0.5

    # Optional: validation data for threshold tuning
    validation_data=None,         # (left_val, right_val, matches_val)

    # Export settings
    base_dir=None,                # defaults to current directory
    export_model=True,
    overwrite=False,              # True replaces a model saved under model_name
    export_stats=True,
    reload_sanity_check=True,     # verify model can be reloaded
)

training_pipeline.execute()

### Probit Model

Probit regression using statsmodels. Similar to Logit but uses the cumulative normal distribution.


In [ ]:
training_pipeline = BaselineTrainingPipe(
    model_name='my_probit_model',
    similarity_map=smap,
    training_data=(left_train, right_train, matches_train),
    testing_data=(left_test, right_test, matches_test),

    model_kind="probit",

    id_left_col="company_id",
    id_right_col="company_id",
    matches_id_left="left",
    matches_id_right="right",
    matches_are_indices=True,

    mismatch_share_fit=1.0,
    threshold=0.5,
    tune_threshold=False,

    export_model=True,
    export_stats=True,
)

training_pipeline.execute()

### Gradient Boosting Model

Gradient boosting with scikit-learn's `HistGradientBoostingClassifier`. More powerful but less interpretable.


In [ ]:
training_pipeline = BaselineTrainingPipe(
    model_name='my_gb_model',
    similarity_map=smap,
    training_data=(left_train, right_train, matches_train),
    testing_data=(left_test, right_test, matches_test),

    model_kind="gb",

    id_left_col="company_id",
    id_right_col="company_id",
    matches_id_left="left",
    matches_id_right="right",
    matches_are_indices=True,

    # Sampling (GB often works well with subsampling)
    mismatch_share_fit=0.5,       # use 50% of non-matches

    # Threshold tuning (recommended for GB)
    tune_threshold=True,          # automatically find best threshold
    tune_metric="mcc",            # "mcc" or "f1"
    validation_data=(left_val, right_val, matches_val),  # required for tuning

    export_model=True,
    export_stats=True,
)

training_pipeline.execute()

The tuned threshold is saved with the model as `best_threshold_`. After
`ModelBaseline.load`, `evaluate` and `GenerateID` use it unless you pass
another one, and `performance.csv` reports it in its `threshold` column, as
it reports 0.5 for Logit and Probit.

Tune only on a validation split that holds enough pairs. Paired record by
record, a split of n records holds about n matches among n² pairs: with a
few dozen records, many thresholds score the same on it, and the one chosen
need not do better on the test pairs than 0.5.

## Deep Learning Model (ANN)

The neural network model uses `TrainingPipe` and supports two-stage training with customizable loss functions.


In [ ]:
from neer_match_utilities.training import TrainingPipe

training_pipeline = TrainingPipe(
    # Required
    model_name='my_ann_model',
    similarity_map=similarity_map,  # dict format, not SimilarityMap object
    training_data=(left_train, right_train, matches_train),
    testing_data=(left_test, right_test, matches_test),

    # Optional: validation data, scored after every epoch
    validation_data=(left_validation, right_validation, matches_validation),
    early_stopping_patience=5,         # stop a stage after 5 epochs without a better score
    tune_threshold=True,               # choose the threshold on the validation pairs (default)
    tune_metric="mcc",                 # "mcc" | "f1"
    tune_match_share="train",          # weight validation non-matches to the training share (default)
    threshold=0.5,                     # used where no threshold is tuned
    evaluation_batch_size=1024,        # pairs per batch when scoring without training

    # ID columns
    id_left_col="company_id",
    id_right_col="company_id",

    # Network architecture
    initial_feature_width_scales=10,  # width multiplier for feature networks
    feature_depths=2,                  # depth of feature networks
    initial_record_width_scale=10,     # width multiplier for record network
    record_depth=4,                    # depth of record network

    # Stage 1: Soft-F1 pretraining
    stage_1=True,
    epochs_1=50,
    mismatch_share_1=0.01,            # fraction of non-matches per epoch
    stage1_loss="soft_f1",            # "soft_f1" | "binary_crossentropy" | callable

    # Stage 2: Focal loss fine-tuning
    stage_2=True,
    epochs_2=30,
    mismatch_share_2=0.1,
    gamma=2.0,                        # focal loss focusing parameter
    max_alpha=0.9,                    # max weight for positive class

    # Batch size control
    no_tm_pbatch=8,                   # target positives per batch

    # Export
    save_architecture=False,          # requires graphviz binaries
    overwrite=False,                  # True replaces a model saved under model_name
)

training_pipeline.execute()

### Key Parameters Explained

**Network Architecture:**

- `initial_feature_width_scales`: Controls the width of feature-specific networks. Higher values create wider networks.
- `feature_depths`: Number of layers in each feature network.
- `initial_record_width_scale`: Controls the width of the final record-comparison network.
- `record_depth`: Number of layers in the record network.
- Keep the layers wide. With small width scales (e.g. 2) and several layers,
  hidden layers of two ReLU units often die and cut fields, or the whole
  network, off from the data; the pipeline warns when a model has collapsed
  this way. The defaults (10, 2, 10, 4) avoid it.

**Training Stages:**

- `stage_1`: Pretraining phase using soft-F1 loss to learn basic matching patterns.
- `stage_2`: Fine-tuning phase using focal loss to focus on hard examples.
- You can disable either stage by setting it to `False`.

**Validation:**

- `validation_data`: If given, the model is scored on all validation pairs after every epoch, and the scores are logged as `val_F1`, `val_MCC`, etc. in the history and in `epoch_overview.csv`.
- `tune_threshold` (default `True`): Each epoch is also scored at the threshold that maximises `tune_metric` on the validation pairs (`val_threshold`, `val_tuned_MCC`), and each stage keeps the epoch with the highest such score. Stage 2 counts stage 1's model as its epoch 0 and keeps it unless one of its own epochs scores higher. The final model's threshold is tuned the same way, used in its final evaluation, saved with it as `best_threshold_` and restored by `Model.load`; `performance.csv` reports it in its `threshold` column. With `tune_threshold=False`, the epochs are chosen by `val_F1` at `threshold`.
- `tune_match_share` (default `"train"`): A small validation split holds a far larger share of matches than the training or test pairs (about 1/n for n records), so that a threshold tuned on it as it is lets through too many false positives. By default the tuning counts its non-matches as if the validation pairs had the training pairs' share of matches; a float sets that share, `None` turns the weighting off. As above, tune only on a validation split that holds enough pairs.
- `early_stopping_patience`: Stops a stage once the score its epochs are chosen by has not improved for this many epochs. Requires `validation_data`.
- `evaluation_batch_size`: Pairs per batch wherever the pipeline scores pairs without training on them. It affects speed and memory, not the scores.

**Sampling:**

- `mismatch_share_1/2`: Fraction of non-matches to sample per epoch. Lower values speed up training but may reduce quality.
- `no_tm_pbatch`: Target number of positive pairs per batch. The actual batch size is calculated automatically.

**Focal Loss (Stage 2):**

- `gamma`: Focusing parameter. Higher values focus more on hard examples (typical: 1.0-3.0).
- `max_alpha`: Maximum class weight for positives. Balances class imbalance.

### Single-Stage Training

You can run only one training stage:


In [ ]:
# Only Stage 1 (faster, simpler)
training_pipeline = TrainingPipe(
    model_name='my_model_stage1_only',
    similarity_map=similarity_map,
    training_data=(left_train, right_train, matches_train),
    testing_data=(left_test, right_test, matches_test),
    id_left_col="company_id",
    id_right_col="company_id",

    stage_1=True,
    epochs_1=100,
    mismatch_share_1=0.05,
    no_tm_pbatch=8,

    stage_2=False,  # Skip stage 2
)

training_pipeline.execute()

## Model Comparison

| Model | Linear | Speed | Interpretability |
|-------|--------|-------|-----------------|
| Logit | Yes | Fast | High |
| Probit | Yes | Fast | High |
| GB | No | Medium | Low |
| ANN | No | Slow | Low |

**Note on performance:** Model performance depends heavily on the specific use case, dataset characteristics, and hyperparameter tuning. ANNs are more prone to getting stuck in local minima, making their results more volatile across runs. Always compare against baseline models for your specific use case.